<a href="https://colab.research.google.com/github/MSaadT313/NLP-SummerCamp-CRAAT/blob/main/5_1_vector_DB_chroma_pinecone_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Database Hands-on Lab: ChromaDB + Optional Pinecone

**Goal:** Learn how a vector database stores embeddings and supports semantic search, metadata filtering, update, delete, and a small Retrieval-Augmented Generation (RAG)-style workflow.

This notebook is designed for **Google Colab**.

## What you will learn

1. Create text embeddings using `sentence-transformers`
2. Store vectors, documents, and metadata in **ChromaDB**
3. Perform:
   - create collection
   - insert/add documents
   - semantic similarity search
   - metadata filtering
   - full-text filtering
   - get/fetch records
   - update records
   - upsert records
   - delete records
   - simple RAG-style retrieval
4. Optional: repeat basic operations with **Pinecone** cloud vector database

## Why vector databases?

Traditional databases search using exact keywords. Vector databases search using meaning.  
For example, the query **"how can students improve their machine learning projects?"** can retrieve a document about **"FYP evaluation and ML experimentation"**, even if the exact words are different.

## 1. Install required libraries

ChromaDB will run locally inside Colab.  
No API key is required for the ChromaDB section.

In [ ]:
!pip install -q --upgrade-strategy eager chromadb sentence-transformers pandas

## 2. Import libraries

In [ ]:
import os
import uuid
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer

## 3. Create a small teaching dataset

Each document has:
- an ID
- text content
- metadata

Metadata is useful when we want to filter search results by category, source, course, semester, etc.

In [ ]:
documents = [
    {
        "id": "doc1",
        "text": "Machine learning models learn patterns from data and make predictions on unseen examples.",
        "metadata": {"topic": "machine_learning", "course": "AI", "level": "intro"}
    },
    {
        "id": "doc2",
        "text": "Natural language processing helps computers understand, analyze, and generate human language.",
        "metadata": {"topic": "nlp", "course": "NLP", "level": "intro"}
    },
    {
        "id": "doc3",
        "text": "Information retrieval systems rank documents according to their relevance to a user query.",
        "metadata": {"topic": "information_retrieval", "course": "IR", "level": "intro"}
    },
    {
        "id": "doc4",
        "text": "Vector databases store embeddings and support similarity search for semantic retrieval.",
        "metadata": {"topic": "vector_database", "course": "AI", "level": "intermediate"}
    },
    {
        "id": "doc5",
        "text": "Retrieval augmented generation first retrieves relevant documents and then uses them as context for generation.",
        "metadata": {"topic": "rag", "course": "NLP", "level": "advanced"}
    },
    {
        "id": "doc6",
        "text": "Evaluation metrics such as precision, recall, F1 score, and MAP are common in information retrieval.",
        "metadata": {"topic": "evaluation", "course": "IR", "level": "intermediate"}
    },
    {
        "id": "doc7",
        "text": "Final year projects should define a clear problem, dataset, baseline model, evaluation metrics, and limitations.",
        "metadata": {"topic": "fyp", "course": "Research", "level": "intro"}
    }
]

df = pd.DataFrame(documents)
df

,id,text,metadata
0,doc1,Machine learning models learn patterns from da...,"{'topic': 'machine_learning', 'course': 'AI', ..."
1,doc2,Natural language processing helps computers un...,"{'topic': 'nlp', 'course': 'NLP', 'level': 'in..."
2,doc3,Information retrieval systems rank documents a...,"{'topic': 'information_retrieval', 'course': '..."
3,doc4,Vector databases store embeddings and support ...,"{'topic': 'vector_database', 'course': 'AI', '..."
4,doc5,Retrieval augmented generation first retrieves...,"{'topic': 'rag', 'course': 'NLP', 'level': 'ad..."
5,doc6,"Evaluation metrics such as precision, recall, ...","{'topic': 'evaluation', 'course': 'IR', 'level..."
6,doc7,Final year projects should define a clear prob...,"{'topic': 'fyp', 'course': 'Research', 'level'..."


## 4. Load an embedding model

We use `sentence-transformers/all-MiniLM-L6-v2`.

It is a lightweight sentence embedding model, suitable for demos and classroom labs.

In [ ]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def embed_texts(texts):
    """Convert a list of text strings into embedding vectors."""
    vectors = model.encode(texts, normalize_embeddings=True)
    return vectors.tolist()

sample_embedding = embed_texts(["Vector databases search by meaning, not only keywords."])[0]

print("Embedding dimension:", len(sample_embedding))
print("First 10 values:", sample_embedding[:10])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384
First 10 values: [0.08839443325996399, -0.02304072305560112, -0.009514833800494671, -0.0030157100409269333, 0.009957289323210716, -0.042170535773038864, 0.03218349814414978, 0.011736415326595306, -0.03106781840324402, -0.0019143014214932919]


## 5. Create a ChromaDB client and collection

A **collection** in ChromaDB is like a table for vectors.  
It stores:
- IDs
- embeddings
- original documents
- metadata

In [ ]:
# A persistent client saves the database files in this Colab runtime folder.
# In Colab, the data will disappear when the runtime is reset unless you save it to Google Drive.
client = chromadb.PersistentClient(path="./chroma_vector_db")

collection_name = "course_notes_demo"

# Re-create the collection for a clean classroom run
try:
    client.delete_collection(collection_name)
except Exception:
    pass

collection = client.create_collection(
    name=collection_name,
    metadata={"description": "Demo collection for vector database teaching", "hnsw:space": "cosine"}
)

print("Collection created:", collection.name)

Collection created: course_notes_demo


## 6. Insert documents into ChromaDB

We manually create embeddings and then add them to the vector database.

In [ ]:
ids = [item["id"] for item in documents]
texts = [item["text"] for item in documents]
metadatas = [item["metadata"] for item in documents]
embeddings = embed_texts(texts)

collection.add(
    ids=ids,
    documents=texts,
    metadatas=metadatas,
    embeddings=embeddings
)

print("Number of records in collection:", collection.count())

Number of records in collection: 7


## 7. Semantic similarity search

Now we ask a natural language question.  
The vector database returns semantically related documents, even if the exact words are not the same.

In [ ]:
query = "How do systems retrieve relevant documents for a user question?"
query_embedding = embed_texts([query])[0]

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

for rank, (doc, meta, dist) in enumerate(
    zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
    start=1
):
    print(f"Rank {rank}")
    print("Distance:", round(dist, 4))
    print("Metadata:", meta)
    print("Document:", doc)
    print("-" * 80)

Rank 1
Distance: 0.4247
Metadata: {'level': 'intro', 'topic': 'information_retrieval', 'course': 'IR'}
Document: Information retrieval systems rank documents according to their relevance to a user query.
--------------------------------------------------------------------------------
Rank 2
Distance: 0.4894
Metadata: {'course': 'NLP', 'level': 'advanced', 'topic': 'rag'}
Document: Retrieval augmented generation first retrieves relevant documents and then uses them as context for generation.
--------------------------------------------------------------------------------
Rank 3
Distance: 0.6642
Metadata: {'course': 'AI', 'topic': 'vector_database', 'level': 'intermediate'}
Document: Vector databases store embeddings and support similarity search for semantic retrieval.
--------------------------------------------------------------------------------


## 8. Metadata filtering

Metadata filters are useful when we want to search only inside a subset.

Example: search only documents where `course = "IR"`.

In [ ]:
query = "ranking documents and measuring retrieval quality"
query_embedding = embed_texts([query])[0]

filtered_results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"course": "IR"},
    include=["documents", "metadatas", "distances"]
)

for rank, (doc, meta, dist) in enumerate(
    zip(filtered_results["documents"][0], filtered_results["metadatas"][0], filtered_results["distances"][0]),
    start=1
):
    print(f"Rank {rank}")
    print("Distance:", round(dist, 4))
    print("Metadata:", meta)
    print("Document:", doc)
    print("-" * 80)

Rank 1
Distance: 0.2294
Metadata: {'topic': 'information_retrieval', 'level': 'intro', 'course': 'IR'}
Document: Information retrieval systems rank documents according to their relevance to a user query.
--------------------------------------------------------------------------------
Rank 2
Distance: 0.3608
Metadata: {'level': 'intermediate', 'course': 'IR', 'topic': 'evaluation'}
Document: Evaluation metrics such as precision, recall, F1 score, and MAP are common in information retrieval.
--------------------------------------------------------------------------------


## 9. Full-text filtering

This is different from semantic search.  
Here we filter documents that contain a specific word or phrase.

In [ ]:
text_filter_results = collection.get(
    where_document={"$contains": "evaluation"},
    include=["documents", "metadatas"]
)

for doc_id, doc, meta in zip(
    text_filter_results["ids"],
    text_filter_results["documents"],
    text_filter_results["metadatas"]
):
    print("ID:", doc_id)
    print("Metadata:", meta)
    print("Document:", doc)
    print("-" * 80)

ID: doc7
Metadata: {'topic': 'fyp', 'level': 'intro', 'course': 'Research'}
Document: Final year projects should define a clear problem, dataset, baseline model, evaluation metrics, and limitations.
--------------------------------------------------------------------------------


## 10. Get/fetch specific records by ID

This is similar to reading rows from a database using primary keys.

In [ ]:
fetched = collection.get(
    ids=["doc2", "doc5"],
    include=["documents", "metadatas"]
)

for doc_id, doc, meta in zip(fetched["ids"], fetched["documents"], fetched["metadatas"]):
    print("ID:", doc_id)
    print("Metadata:", meta)
    print("Document:", doc)
    print("-" * 80)

ID: doc2
Metadata: {'topic': 'nlp', 'course': 'NLP', 'level': 'intro'}
Document: Natural language processing helps computers understand, analyze, and generate human language.
--------------------------------------------------------------------------------
ID: doc5
Metadata: {'level': 'advanced', 'topic': 'rag', 'course': 'NLP'}
Document: Retrieval augmented generation first retrieves relevant documents and then uses them as context for generation.
--------------------------------------------------------------------------------


## 11. Update an existing record

Suppose we want to update `doc4` with a better explanation.

In [ ]:
updated_text = "A vector database stores numerical embeddings and quickly finds items with similar meaning."
updated_metadata = {"topic": "vector_database", "course": "AI", "level": "intermediate", "updated": True}
updated_embedding = embed_texts([updated_text])[0]

collection.update(
    ids=["doc4"],
    documents=[updated_text],
    metadatas=[updated_metadata],
    embeddings=[updated_embedding]
)

updated_record = collection.get(
    ids=["doc4"],
    include=["documents", "metadatas"]
)

print(updated_record)

{'ids': ['doc4'], 'embeddings': None, 'documents': ['A vector database stores numerical embeddings and quickly finds items with similar meaning.'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'updated': True, 'level': 'intermediate', 'topic': 'vector_database', 'course': 'AI'}]}


## 12. Upsert a record

`upsert` means:
- insert the record if it does not exist
- update it if it already exists

This is useful in production pipelines where documents may be repeatedly refreshed.

In [ ]:
new_doc_id = "doc8"
new_text = "Hybrid search combines keyword search with vector similarity search for better retrieval."
new_metadata = {"topic": "hybrid_search", "course": "IR", "level": "advanced"}
new_embedding = embed_texts([new_text])[0]

collection.upsert(
    ids=[new_doc_id],
    documents=[new_text],
    metadatas=[new_metadata],
    embeddings=[new_embedding]
)

print("Number of records after upsert:", collection.count())

record = collection.get(ids=[new_doc_id], include=["documents", "metadatas"])
print(record)

Number of records after upsert: 8
{'ids': ['doc8'], 'embeddings': None, 'documents': ['Hybrid search combines keyword search with vector similarity search for better retrieval.'], 'uris': None, 'included': ['documents', 'metadatas'], 'data': None, 'metadatas': [{'level': 'advanced', 'course': 'IR', 'topic': 'hybrid_search'}]}


## 13. Delete a record

We can remove vectors/documents by ID.

In [ ]:
collection.delete(ids=["doc8"])

print("Number of records after deleting doc8:", collection.count())

Number of records after deleting doc8: 7


## 14. Simple RAG-style retrieval function

In a real RAG system, the retrieved context is passed to a language model.  
Here, we only show the retrieval step and construct a simple prompt.

In [ ]:
def retrieve_context(question, top_k=3, course_filter=None):
    query_embedding = embed_texts([question])[0]

    query_args = {
        "query_embeddings": [query_embedding],
        "n_results": top_k,
        "include": ["documents", "metadatas", "distances"]
    }

    if course_filter is not None:
        query_args["where"] = {"course": course_filter}

    results = collection.query(**query_args)

    retrieved_docs = results["documents"][0]
    retrieved_metas = results["metadatas"][0]
    retrieved_distances = results["distances"][0]

    return list(zip(retrieved_docs, retrieved_metas, retrieved_distances))


question = "What should be included in a good final year project?"
contexts = retrieve_context(question, top_k=3)

print("Question:", question)
print("\nRetrieved Contexts:")
for i, (doc, meta, dist) in enumerate(contexts, start=1):
    print(f"Context {i} | Distance: {round(dist, 4)} | Metadata: {meta}")
    print(doc)
    print("-" * 80)

prompt = f'''
Answer the question using only the context below.

Question:
{question}

Context:
{chr(10).join([f"- {doc}" for doc, _, _ in contexts])}

Answer:
'''

print("\nGenerated RAG-style prompt:")
print(prompt)

Question: What should be included in a good final year project?

Retrieved Contexts:
Context 1 | Distance: 0.2842 | Metadata: {'level': 'intro', 'course': 'Research', 'topic': 'fyp'}
Final year projects should define a clear problem, dataset, baseline model, evaluation metrics, and limitations.
--------------------------------------------------------------------------------
Context 2 | Distance: 0.8471 | Metadata: {'topic': 'rag', 'level': 'advanced', 'course': 'NLP'}
Retrieval augmented generation first retrieves relevant documents and then uses them as context for generation.
--------------------------------------------------------------------------------
Context 3 | Distance: 0.8775 | Metadata: {'topic': 'machine_learning', 'course': 'AI', 'level': 'intro'}
Machine learning models learn patterns from data and make predictions on unseen examples.
--------------------------------------------------------------------------------

Generated RAG-style prompt:

Answer the question using on

## 15. Show all collections

This helps us inspect available collections in the local ChromaDB client.

In [ ]:
collections = client.list_collections()
collections

[Collection(name=course_notes_demo)]

# Optional Section: Pinecone Cloud Vector Database

Use this section only if you have a Pinecone API key.

## What is different?

ChromaDB above runs locally in Colab.  
Pinecone is a managed cloud vector database, useful when you want:
- hosted infrastructure
- production-scale indexes
- namespaces for separating tenants/classes/projects
- cloud deployment

## Setup notes

1. Create a Pinecone account.
2. Create/get your API key.
3. In Colab, go to the left sidebar:
   - click the key icon **Secrets**
   - add a secret named `PINECONE_API_KEY`
4. Then run the cells below.

In [ ]:
# Optional install for Pinecone
!pip -q install pinecone

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 15.8 MB/s eta 0:00:00


## 16. Connect to Pinecone

This cell reads the API key from Colab Secrets.

If you are not using Colab Secrets, you can set:
`os.environ["PINECONE_API_KEY"] = "your-key-here"`

Do not hard-code real API keys in notebooks you share with students.

In [ ]:
# OPTIONAL: Run only when you have a Pinecone API key

try:
    from google.colab import userdata
    PINECONE_API_KEY = userdata.get("Demo")
except Exception:
    PINECONE_API_KEY = os.environ.get("Demo")

print("Pinecone key loaded:", bool(PINECONE_API_KEY))

Pinecone key loaded: True


## 17. Create a Pinecone index

The embedding dimension must match the embedding model.

For `sentence-transformers/all-MiniLM-L6-v2`, the dimension is usually **384**.

In [ ]:
# OPTIONAL: Run only when PINECONE_API_KEY is available

if PINECONE_API_KEY:
    from pinecone import Pinecone, ServerlessSpec

    pc = Pinecone(api_key=PINECONE_API_KEY)

    index_name = "vector-db-demo"
    dimension = len(sample_embedding)

    existing_indexes = [index_info["name"] for index_info in pc.list_indexes()]

    if index_name not in existing_indexes:
        pc.create_index(
            name=index_name,
            dimension=dimension,
            metric="cosine",
            spec=ServerlessSpec(
                cloud="aws",
                region="us-east-1"
            )
        )
        print("Created Pinecone index:", index_name)
    else:
        print("Index already exists:", index_name)

    pinecone_index = pc.Index(index_name)
else:
    print("Skipping Pinecone index creation because API key is not available.")

Index already exists: vector-db-demo


## 18. Upsert vectors into Pinecone

Each vector has:
- ID
- values
- metadata

We use a namespace named `teaching-demo`.

In [ ]:
# OPTIONAL: Run only when PINECONE_API_KEY is available

if PINECONE_API_KEY:
    vectors_to_upsert = []

    for item, emb in zip(documents, embeddings):
        vectors_to_upsert.append({
            "id": item["id"],
            "values": emb,
            "metadata": {
                **item["metadata"],
                "text": item["text"]
            }
        })

    pinecone_index.upsert(
        vectors=vectors_to_upsert,
        namespace="teaching-demo"
    )

    print("Upserted records into Pinecone.")
else:
    print("Skipping Pinecone upsert because API key is not available.")

Upserted records into Pinecone.


## 19. Query Pinecone

In [ ]:
# OPTIONAL: Run only when PINECONE_API_KEY is available

if PINECONE_API_KEY:
    q = "semantic retrieval using vector embeddings"
    q_vec = embed_texts([q])[0]

    pinecone_results = pinecone_index.query(
        vector=q_vec,
        top_k=3,
        namespace="teaching-demo",
        include_metadata=True
    )

    for match in pinecone_results["matches"]:
        print("ID:", match["id"])
        print("Score:", round(match["score"], 4))
        print("Metadata:", match["metadata"])
        print("-" * 80)
else:
    print("Skipping Pinecone query because API key is not available.")

ID: doc4
Score: 0.8364
Metadata: {'course': 'AI', 'level': 'intermediate', 'text': 'Vector databases store embeddings and support similarity search for semantic retrieval.', 'topic': 'vector_database'}
--------------------------------------------------------------------------------
ID: doc3
Score: 0.4693
Metadata: {'course': 'IR', 'level': 'intro', 'text': 'Information retrieval systems rank documents according to their relevance to a user query.', 'topic': 'information_retrieval'}
--------------------------------------------------------------------------------
ID: doc6
Score: 0.4346
Metadata: {'course': 'IR', 'level': 'intermediate', 'text': 'Evaluation metrics such as precision, recall, F1 score, and MAP are common in information retrieval.', 'topic': 'evaluation'}
--------------------------------------------------------------------------------


## 20. Delete vectors from Pinecone

In [ ]:
# OPTIONAL: Run only when PINECONE_API_KEY is available

if PINECONE_API_KEY:
    pinecone_index.delete(
        ids=["doc7"],
        namespace="teaching-demo"
    )
    print("Deleted doc7 from Pinecone namespace teaching-demo.")
else:
    print("Skipping Pinecone delete because API key is not available.")

Deleted doc7 from Pinecone namespace teaching-demo.


# Summary of Vector Database Operations

| Operation | Meaning | Example use |
|---|---|---|
| Create collection/index | Create storage for embeddings | Course notes database |
| Add/Insert | Store new documents and vectors | Add lecture notes |
| Query/Search | Find similar vectors | Semantic search |
| Metadata filter | Restrict search by metadata | Search only NLP documents |
| Full-text filter | Search by exact text condition | Find documents containing "evaluation" |
| Get/Fetch | Retrieve known records by ID | Read `doc2` |
| Update | Modify an existing record | Correct a document |
| Upsert | Insert or update | Refresh document pipeline |
| Delete | Remove records | Remove outdated content |
| RAG retrieval | Fetch context for LLM answer | Chatbot over course material |

## Suggested classroom extension

Ask students to replace the toy documents with:
1. course lecture notes,
2. research paper abstracts,
3. university policy documents,
4. FYP proposal summaries,
5. product descriptions for recommendation/search.